# Multilingual Audio Processing with Language Detection and Translation

This notebook processes audio files containing multiple languages (Telugu, Hindi, English), detects each word's language, and translates them to a target language.

## Import Required Libraries

In [25]:
import os
import whisper
from pydub import AudioSegment
from pydub.silence import split_on_silence
from deep_translator import GoogleTranslator
from langdetect import detect

# Options: tiny, base, small, medium, large
whisper_model = whisper.load_model("small")
print("Whisper model loaded successfully!")

Whisper model loaded successfully!


In [26]:
# Verify FFmpeg is accessible from system PATH
import subprocess
import shutil

# Check if ffmpeg and ffprobe are in PATH
ffmpeg_path = shutil.which("ffmpeg")
ffprobe_path = shutil.which("ffprobe")

print("FFmpeg found in PATH:", ffmpeg_path if ffmpeg_path else "Not found")
print("FFprobe found in PATH:", ffprobe_path if ffprobe_path else "Not found")

# Test FFmpeg functionality
if ffmpeg_path:
    try:
        result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
        print("\nFFmpeg is working correctly!")
        print("Version:", result.stdout.split('\n')[0])
    except Exception as e:
        print(f"\nError testing FFmpeg: {e}")
else:
    print("\nWarning: FFmpeg not found in PATH. Please restart your terminal/notebook.")

FFmpeg found in PATH: C:\Program Files\ffmpeg\bin\ffmpeg.EXE
FFprobe found in PATH: C:\Program Files\ffmpeg\bin\ffprobe.EXE

FFmpeg is working correctly!
Version: ffmpeg version N-121490-g7e8ef2ded2-20251024 Copyright (c) 2000-2025 the FFmpeg developers


## Function 1: Split Audio into Segments

This function splits the audio file into individual segments based on silence detection.

In [27]:
def audio_to_segments(audio_path):
    """Split audio file into segments based on silence"""
    audio = AudioSegment.from_file(audio_path)
    
    # Split audio on silence (adjust parameters as needed)
    segments = split_on_silence(
        audio,
        min_silence_len=500,  # minimum silence length in ms
        silence_thresh=audio.dBFS - 14,  # silence threshold
        keep_silence=250  # keep some silence at edges
    )
    
    return segments

## Function 2: Detect Language from Text

This function detects the language of transcribed text.

In [28]:
def detect_language_from_text(text):
    """Detect language from text using script analysis and langdetect with smart mapping"""
    try:
        # First check if it contains Devanagari script (Hindi)
        devanagari_chars = sum(1 for c in text if '\u0900' <= c <= '\u097F')
        if devanagari_chars > 0:
            return 'hindi'
        
        # Check if it contains Telugu script
        telugu_chars = sum(1 for c in text if '\u0C00' <= c <= '\u0C7F')
        if telugu_chars > 0:
            return 'telugu'
        
        # For romanized text, use langdetect with smart mapping
        # langdetect often misidentifies romanized Indian languages
        lang = detect(text)
        
        # Map common false positives for romanized Indian languages
        lang_map = {
            'en': 'english',
            'hi': 'hindi',
            'te': 'telugu',
            'ta': 'telugu',
            'ro': 'hindi',  # Romanian is often misdetected Hindi romanized
            'it': 'telugu',  # Italian is often misdetected Telugu romanized
            'id': 'telugu',  # Indonesian is often misdetected Telugu romanized
            'pt': 'hindi',  # Portuguese sometimes misdetected as Hindi romanized
        }
        
        return lang_map.get(lang, 'english')  # Default to English
    except:
        return 'english'

## Function 3: Transcribe Audio Segment

This function transcribes a single audio segment in multiple languages.

In [29]:
def transcribe_audio_segment(audio_segment):
    """Transcribe a single audio segment in English using Whisper AI"""
    import time
    
    # Normalize audio volume
    audio_segment = audio_segment.normalize()
    
    # Export segment to temporary file
    temp_file = f"temp_segment_{time.time()}.wav"
    audio_segment.export(
        temp_file, 
        format="wav",
        parameters=["-ar", "16000", "-ac", "1"]  # 16kHz sample rate, mono
    )
    
    try:
        # Transcribe in English
        result = whisper_model.transcribe(
            temp_file,
            language='en',
            task='transcribe',
            fp16=False
        )
        english_text = result['text'].strip()
        
        if english_text:
            print(f"    English: {english_text}")
            return english_text
        else:
            return None
                    
    except Exception as e:
        print(f"    Transcription error: {str(e)[:100]}")
        return None
                
    finally:
        # Clean up temp file
        try:
            time.sleep(0.1)
            if os.path.exists(temp_file):
                os.remove(temp_file)
        except:
            pass

## Function 4: Translate Text

This function translates text from source language to target language.

In [30]:
def translate_text(text, source_lang, target_lang):
    """Translate text from source to target language"""
    lang_codes = {
        'english': 'en',
        'hindi': 'hi',
        'telugu': 'te'
    }
    
    try:
        src_code = lang_codes.get(source_lang, 'auto')
        dest_code = lang_codes.get(target_lang, 'en')
        
        translation = GoogleTranslator(source=src_code, target=dest_code).translate(text)
        return translation
    except Exception as e:
        print(f"Translation error: {e}")
        return text

## Function 5: Main Processing Function

This is the main function that orchestrates the entire process.

In [31]:
def process_english_audio(audio_path):
    """Main function to process English audio"""
    print(f"Processing audio file: {audio_path}\n")
    
    # Split audio into segments
    segments = audio_to_segments(audio_path)
    print(f"Found {len(segments)} segments\n")
    
    results = []
    
    for i, segment in enumerate(segments):
        print(f"Processing segment {i+1}/{len(segments)}...")
        
        # Transcribe segment in English
        english_text = transcribe_audio_segment(segment)
        
        if not english_text:
            print(f"  No speech detected in segment {i+1}\n")
            continue
        
        results.append({
            'segment': i+1,
            'english_text': english_text
        })
        print()
    
    return results

## Example Usage

Run this cell to process your audio file. Make sure to replace the path with your actual audio file.

In [36]:
# Example usage
audio_file = "audio_files/cleaned_audio.wav" 

# Choose target language: 'telugu' or 'hindi'
target_language = 'telugu'  # Change to 'hindi' if needed

# Process English audio and translate to target language
results = process_english_audio(audio_file)

# Display results with translations
print("\n" + "="*50)
print(f"RESULTS (English → {target_language.upper()})")
print("="*50 + "\n")
for result in results:
    print(f"Segment {result['segment']}:")
    print(f"  English: {result['english_text']}")
    # Translate to target language
    translated_text = translate_text(result['english_text'], 'english', target_language)
    print(f"  {target_language.capitalize()}: {translated_text}")
    print()

# Create complete transcript in target language
print("\n" + "="*50)
print(f"COMPLETE TRANSCRIPT ({target_language.upper()})")
print("="*50 + "\n")
complete_transcript_english = " ".join([result['english_text'] for result in results])
complete_transcript_translated = translate_text(complete_transcript_english, 'english', target_language)
print(complete_transcript_translated)

Processing audio file: audio_files/cleaned_audio.wav

Found 135 segments

Processing segment 1/135...
    English: This is it.

Processing segment 2/135...
    English: Sweet.

Processing segment 3/135...
    English: Do.

Processing segment 4/135...
    English: What?

Processing segment 5/135...
    English: stuff.

Processing segment 6/135...
    English: Alien stood in the center of the ship.

Processing segment 7/135...
    English: that's

Processing segment 8/135...
    English: It's just swelling.

Processing segment 9/135...
    English: Right?

Processing segment 10/135...
    English: so dense it felt like

Processing segment 11/135...
    English: After 10 years of possession of sleepless nights and a year back.

Processing segment 12/135...
    English: the Skyline Project was launched.

Processing segment 13/135...
    English: it was its possibleness.

Processing segment 14/135...
    English: If we don't...

Processing segment 15/135...
    English: his hands trembling 

KeyboardInterrupt: 

## Concatenated Transcript

This cell concatenates all segments and displays the complete transcript.

In [34]:
# Display complete English transcript
print("\n" + "="*50)
print("COMPLETE TRANSCRIPT (English)")
print("="*50 + "\n")

complete_transcript_english = " ".join([result['english_text'] for result in results])
print(complete_transcript_english)

# Display complete translated transcript
print("\n" + "="*50)
print(f"COMPLETE TRANSCRIPT ({target_language.upper()})")
print("="*50 + "\n")

complete_transcript_translated = translate_text(complete_transcript_english, 'english', target_language)
print(complete_transcript_translated)


COMPLETE TRANSCRIPT (English)

Yeah, hello, hello, yes. Look at three. 2, 1, start. Elea stood in the center of shimmering glass atrium. His chest swelling with pride so intense it felt like helium. After 10 years of rejection, sleepless nights and a gear back cramp see The Skyline project was finished. It was his masterpiece. He pulled out his phone. his hands trembling with excitement and dialed the one person who had believed in him when no one else did. is and Strangle brother Julian. Julian. Elias Beamed. his voice echoed through in the empty hall. I did it. The building opens tomorrow. I want you there. I'm finally going to pay you back every cent you let me. With interest. We made it brother. There was a long silence on the other end. Then... a voice that wasn't Julian's. Spook Yeah You It was trembling and clinical. Mr. Vance? This is... Berry's Hospital We have found this form in the personal effects of a patient admitted an hour ago. I'm sorry. Julien suffered a massive card

## Installation Instructions

Before running this notebook, you need to install the required packages. Run the following command in your terminal:

```bash
pip install openai-whisper pydub deep-translator langdetect numpy
```

You'll also need to install **FFmpeg** for audio processing:
- **Windows**: Download from https://ffmpeg.org/ and add to PATH
- **Linux**: `sudo apt-get install ffmpeg`
- **Mac**: `brew install ffmpeg`

**Note**: This notebook uses Whisper AI, a free and open-source model from OpenAI that provides highly accurate multilingual speech recognition. The first time you run it, Whisper will download the model file (approximately 140 MB for the base model).